# 🔍 Interprétabilité des Modèles - Démo Complète

**Objectif:** Comprendre les décisions des modèles de Deep Learning avec 3 techniques complémentaires

## 📋 Techniques d'interprétabilité

1. **Grad-CAM** (Gradient-weighted Class Activation Mapping)
   - Visualise les zones d'attention du modèle
   - Basé sur les gradients des couches convolutionnelles
   - Rapide et efficace pour CNN

2. **LIME** (Local Interpretable Model-agnostic Explanations)
   - Explique par segmentation d'image (super-pixels)
   - Modèle agnostique (fonctionne avec tout modèle)
   - Bonne pour comprendre les régions importantes

3. **SHAP** (SHapley Additive exPlanations)
   - Basé sur la théorie des jeux (valeurs de Shapley)
   - Explications au niveau pixel
   - Théoriquement fondé

## 🎯 Dataset

COVID-19 Radiography (4 classes):
- COVID
- Normal  
- Lung_Opacity
- Viral Pneumonia

## 📦 Installation des dépendances

Si vous n'avez pas installé les packages d'interprétabilité:

```bash
pip install lime shap scikit-image
```

In [ ]:
# Imports standard
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import tensorflow as tf
from tensorflow import keras

# Imports du projet
import sys
project_root = Path.cwd().parent.parent  # De notebooks/ vers racine du projet
# sys.path.insert(0, str(project_root))

from src.interpretability import (
    GradCAM, visualize_gradcam,
    LIMEImageExplainer,
    SHAPExplainer,
    plot_multiple_explanations,
    create_interpretation_report
)

# Configuration matplotlib
plt.rcParams['figure.figsize'] = (15, 5)
sns.set_style('whitegrid')

print("✅ Imports réussis")
print(f"📂 Project root: {project_root}")

ModuleNotFoundError: No module named 'src'

## 1️⃣ Chargement du Modèle et des Données

In [ ]:
# Chemins
models_dir = project_root / 'models'
data_dir = project_root / 'data' / 'raw' / 'COVID-19_Radiography_Dataset' / 'COVID-19_Radiography_Dataset'
results_dir = project_root / 'results' / 'interpretability'
results_dir.mkdir(parents=True, exist_ok=True)

# Classes
categories = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
n_classes = len(categories)

print(f"📂 Modèles: {models_dir}")
print(f"📂 Données: {data_dir}")
print(f"📂 Résultats: {results_dir}")
print(f"🏷️ Classes: {categories}")

In [ ]:
# Vérifier si un modèle existe déjà
model_path = models_dir / 'interpretability_inceptionv3.keras'

if model_path.exists():
    print(f"✅ Modèle trouvé, chargement...")
    model = keras.models.load_model(model_path)
    print(f"   Modèle: {model_path.name}")
    print(f"   Input shape: {model.input_shape}")
    print(f"   Output shape: {model.output_shape}")
else:
    print(f"⚠️ Aucun modèle trouvé, entraînement d'un nouveau modèle InceptionV3...")
    print(f"   Cela peut prendre 10-20 minutes selon votre GPU")

## 2️⃣ Préparation des Données de Test

In [ ]:
# Charger quelques images de test
from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
from src.features.Pipelines.Transformateurs.image_preprocessing import ImageResizer, ImageNormalizer
from sklearn.pipeline import Pipeline

# Pipeline de preprocessing (identique à l'entraînement)
prep_pipeline = Pipeline([
    ('load', ImageLoader(color_mode='RGB', verbose=False)),
    ('resize', ImageResizer(img_size=(224, 224), verbose=False)),
    ('norm', ImageNormalizer(method='minmax', verbose=False))
])

# Charger 5 images par classe (20 total)
n_images_per_class = 5
test_images = []
test_labels = []
test_paths = []

for idx, cat in enumerate(categories):
    cat_path = data_dir / cat / 'images'
    imgs = sorted(list(cat_path.glob('*.png')))[:n_images_per_class]
    test_paths.extend(imgs)
    test_labels.extend([idx] * len(imgs))

# Préprocessing
test_images = prep_pipeline.fit_transform(test_paths)
test_labels = np.array(test_labels)

print(f"✅ {len(test_images)} images de test chargées")
print(f"   Shape: {test_images.shape}")
print(f"   Distribution: {np.bincount(test_labels)}")

In [ ]:
# Prédictions
predictions = model.predict(test_images, verbose=0)
pred_labels = np.argmax(predictions, axis=1)
confidences = np.max(predictions, axis=1)

# Accuracy
accuracy = np.mean(pred_labels == test_labels)
print(f"\n📊 Résultats:")
print(f"   Accuracy: {accuracy:.2%}")
print(f"   Correct: {np.sum(pred_labels == test_labels)}/{len(test_labels)}")
print(f"   Confiance moyenne: {np.mean(confidences):.2%}")

## 3️⃣ Grad-CAM - Visualisation des Zones d'Attention

In [ ]:
# Créer l'explainer Grad-CAM
gradcam = GradCAM(model)

# Afficher les couches convolutionnelles disponibles
conv_layers = gradcam.get_available_layers()
print(f"Couches convolutionnelles disponibles: {len(conv_layers)}")
for layer in conv_layers[-5:]:  # Dernières 5 couches
    print(f"  • {layer}")

print(f"\n✅ Utilisation de la couche: {gradcam.layer_name}")

In [ ]:
# Exemple 1: Visualiser Grad-CAM pour une image COVID
covid_idx = 0  # Première image COVID
image = test_images[covid_idx]
true_label = test_labels[covid_idx]
pred_label = pred_labels[covid_idx]
confidence = confidences[covid_idx]

# Calculer la heatmap
heatmap = gradcam.compute_heatmap(image, class_idx=pred_label)

# Visualiser
fig = visualize_gradcam(
    image,
    heatmap,
    class_name=categories[pred_label],
    confidence=confidence,
    save_path=results_dir / 'gradcam_covid_example.png'
)
plt.show()

print(f"Vérité: {categories[true_label]}")
print(f"Prédiction: {categories[pred_label]} ({confidence:.2%})")

In [ ]:
# Exemple 2: Comparer les heatmaps pour toutes les classes
from src.interpretability.gradcam import compare_layers

# Prendre une image mal classée (si disponible)
incorrect_indices = np.where(pred_labels != test_labels)[0]

if len(incorrect_indices) > 0:
    idx = incorrect_indices[0]
    image = test_images[idx]
    
    print(f"Image mal classée:")
    print(f"  Vérité: {categories[test_labels[idx]]}")
    print(f"  Prédiction: {categories[pred_labels[idx]]} ({confidences[idx]:.2%})")
    
    # Comparer les couches
    fig = compare_layers(
        model,
        image,
        layer_names=conv_layers[-4:],  # 4 dernières couches
        class_idx=pred_labels[idx],
        save_path=results_dir / 'gradcam_layer_comparison.png'
    )
    plt.show()
else:
    print("✅ Toutes les prédictions sont correctes!")

In [ ]:
# Exemple 3: Grille de visualisations Grad-CAM
from src.interpretability.gradcam import visualize_gradcam_grid

# Sélectionner 6 images diverses
selected_indices = [0, 5, 10, 15, 18, 19]
selected_images = test_images[selected_indices]
selected_labels = pred_labels[selected_indices]
selected_confidences = confidences[selected_indices]

# Calculer les heatmaps
heatmaps = []
for i in selected_indices:
    heatmap = gradcam.compute_heatmap(test_images[i], class_idx=pred_labels[i])
    heatmaps.append(heatmap)

# Visualiser en grille
class_names_selected = [categories[label] for label in selected_labels]

fig = visualize_gradcam_grid(
    selected_images,
    heatmaps,
    class_names_selected,
    confidences=selected_confidences,
    n_cols=3,
    figsize=(18, 12),
    save_path=results_dir / 'gradcam_grid.png'
)
plt.show()

## 4️⃣ LIME - Explication par Super-pixels

In [ ]:
# Créer l'explainer LIME
lime_explainer = LIMEImageExplainer(
    model.predict,
    segmentation_method='quickshift',  # 'quickshift', 'felzenszwalb', 'slic'
    num_samples=1000
)

print("✅ LIME Explainer créé")
print(f"   Méthode de segmentation: quickshift")
print(f"   Nombre d'échantillons: 1000")

In [ ]:
# Exemple 1: Explication LIME pour une image
idx = 0  # Première image
image = test_images[idx]
pred_label = pred_labels[idx]

print(f"Génération de l'explication LIME...")
print(f"  Classe: {categories[pred_label]}")

# Générer l'explication (peut prendre 30-60 secondes)
explanation = lime_explainer.explain_instance(
    image,
    top_labels=1,
    num_features=10
)

# Visualiser
fig = lime_explainer.visualize_explanation(
    image,
    explanation,
    label=pred_label,
    positive_only=True,
    num_features=5,
    save_path=results_dir / 'lime_example.png'
)
plt.show()

print("✅ Explication LIME générée")

In [ ]:
# Exemple 2: Visualisation avec frontières de super-pixels
fig = lime_explainer.visualize_explanation_boundaries(
    image,
    explanation,
    label=pred_label,
    num_features=5,
    save_path=results_dir / 'lime_boundaries.png'
)
plt.show()

In [ ]:
# Exemple 3: Contributions des features
fig = lime_explainer.visualize_top_features(
    explanation,
    label=pred_label,
    num_features=10,
    save_path=results_dir / 'lime_features.png'
)
plt.show()

In [ ]:
# Exemple 4: Comparaison des méthodes de segmentation
fig = lime_explainer.compare_segmentation_methods(
    image,
    methods=['quickshift', 'felzenszwalb', 'slic'],
    save_path=results_dir / 'lime_segmentation_comparison.png'
)
plt.show()

## 5️⃣ SHAP - Valeurs de Shapley

In [ ]:
# Créer l'explainer SHAP
# IMPORTANT: SHAP nécessite des données de référence (background)
# Utiliser un subset du training set (50-100 images)

print("Préparation des données de référence pour SHAP...")

# Charger 50 images aléatoires (10 par classe)
n_background = 10
background_images = []
background_labels = []

for idx, cat in enumerate(categories):
    cat_path = data_dir / cat / 'images'
    imgs = sorted(list(cat_path.glob('*.png')))[100:100+n_background]  # Éviter overlap avec test
    background_images.extend(imgs)
    background_labels.extend([idx] * len(imgs))

# Préprocessing
background_data = prep_pipeline.transform(background_images)
background_labels = np.array(background_labels)

print(f"✅ {len(background_data)} images de référence chargées")
print(f"   Distribution: {np.bincount(background_labels)}")

In [ ]:
# Créer l'explainer SHAP
shap_explainer = SHAPExplainer(model, background_data)

print("✅ SHAP Explainer créé")

In [ ]:
# Exemple 1: Calculer les valeurs SHAP pour quelques images
# ATTENTION: SHAP est plus lent que Grad-CAM/LIME
# Commencer avec 3-5 images

n_shap_images = 3
indices = [0, 5, 10]  # 3 images de classes différentes

print(f"Calcul des valeurs SHAP pour {n_shap_images} images...")
print("⏳ Cela peut prendre plusieurs minutes...")

shap_images = test_images[indices]
shap_values = shap_explainer.explain(shap_images, check_additivity=False)

print("✅ Valeurs SHAP calculées")

In [ ]:
# Visualiser la première explication
idx = 0
image = shap_images[idx]
pred_label = pred_labels[indices[idx]]

fig = shap_explainer.visualize_image_plot(
    image,
    shap_values[idx],
    class_idx=pred_label,
    class_name=categories[pred_label],
    save_path=results_dir / 'shap_example.png'
)
plt.show()

In [ ]:
# Exemple 2: Heatmap SHAP superposée
fig = shap_explainer.visualize_heatmap(
    image,
    shap_values[idx],
    class_idx=pred_label,
    save_path=results_dir / 'shap_heatmap.png'
)
plt.show()

In [ ]:
# Exemple 3: Comparer les SHAP values pour toutes les classes
fig = shap_explainer.compare_classes(
    image,
    shap_values[idx],
    categories,
    save_path=results_dir / 'shap_classes_comparison.png'
)
plt.show()

## 6️⃣ Comparaison des 3 Méthodes

In [ ]:
# Comparer les 3 méthodes sur la même image
idx = 0  # Première image
image = test_images[idx]
pred_label = pred_labels[idx]
confidence = confidences[idx]

print(f"Génération des 3 explications pour l'image {idx}...")
print(f"  Classe prédite: {categories[pred_label]} ({confidence:.2%})")

# Grad-CAM
gradcam_heatmap = gradcam.compute_heatmap(image, class_idx=pred_label)

# LIME
lime_explanation = lime_explainer.explain_instance(image, top_labels=1, num_features=5)

# SHAP
shap_single = shap_explainer.explain(image[np.newaxis, ...], check_additivity=False)
shap_values_single = shap_single[0][pred_label]

# Visualiser côte à côte
fig = plot_multiple_explanations(
    image,
    gradcam_heatmap=gradcam_heatmap,
    lime_explanation=lime_explanation,
    shap_values=shap_values_single,
    class_idx=pred_label,
    class_name=categories[pred_label],
    confidence=confidence,
    save_path=results_dir / 'comparison_all_methods.png'
)
plt.show()

print("✅ Comparaison générée")

## 7️⃣ Rapport Complet d'Interprétabilité

In [ ]:
# Générer un rapport complet pour une image
idx = 0
image = test_images[idx]
true_label = test_labels[idx]
pred_label = pred_labels[idx]
confidence = confidences[idx]

print(f"Génération du rapport complet...")
print(f"  Image: {idx}")
print(f"  Vérité: {categories[true_label]}")
print(f"  Prédiction: {categories[pred_label]} ({confidence:.2%})")

report = create_interpretation_report(
    image,
    model,
    categories,
    true_label,
    pred_label,
    confidence,
    save_dir=results_dir / 'reports',
    filename_prefix=f"image_{idx:03d}",
    include_lime=True,
    include_shap=True,
    background_data=background_data
)

print("\n✅ Rapport complet généré")
print(f"   Dossier: {results_dir / 'reports'}")

## 8️⃣ Analyse des Erreurs avec Interprétabilité

In [ ]:
# Analyser les images mal classées
incorrect_indices = np.where(pred_labels != test_labels)[0]

if len(incorrect_indices) > 0:
    print(f"❌ {len(incorrect_indices)} erreurs de classification trouvées")
    
    for i, idx in enumerate(incorrect_indices[:3]):  # 3 premières erreurs
        print(f"\n{'='*70}")
        print(f"ERREUR {i+1}/{len(incorrect_indices[:3])}")
        print(f"{'='*70}")
        
        image = test_images[idx]
        true_label = test_labels[idx]
        pred_label = pred_labels[idx]
        confidence = confidences[idx]
        
        print(f"  Vérité: {categories[true_label]}")
        print(f"  Prédiction: {categories[pred_label]} ({confidence:.2%})")
        
        # Grad-CAM
        heatmap = gradcam.compute_heatmap(image, class_idx=pred_label)
        
        # Visualiser
        fig = visualize_gradcam(
            image,
            heatmap,
            class_name=f"Erreur: prédit {categories[pred_label]}, vrai {categories[true_label]}",
            confidence=confidence,
            save_path=results_dir / f'error_analysis_{i+1}.png'
        )
        plt.show()
else:
    print("✅ Aucune erreur de classification!")

## 9️⃣ Métriques d'Interprétabilité

In [ ]:
from src.interpretability.utils import (
    compute_explanation_metrics,
    compare_explanation_metrics,
    visualize_metrics_comparison
)

# Comparer les métriques pour une image
idx = 0
image = test_images[idx]
pred_label = pred_labels[idx]

# Grad-CAM
gradcam_heatmap = gradcam.compute_heatmap(image, class_idx=pred_label)

# LIME (convertir en heatmap)
lime_exp = lime_explainer.explain_instance(image, top_labels=1, num_features=5)
_, lime_mask = lime_exp.get_image_and_mask(pred_label, positive_only=True, num_features=5, hide_rest=False)

# SHAP
shap_vals = shap_explainer.explain(image[np.newaxis, ...], check_additivity=False)
shap_heatmap = np.mean(np.abs(shap_vals[0][pred_label]), axis=-1)

# Comparer
metrics = compare_explanation_metrics(
    gradcam_heatmap=gradcam_heatmap,
    lime_mask=lime_mask,
    shap_heatmap=shap_heatmap,
    image=image
)

# Visualiser
fig = visualize_metrics_comparison(
    metrics,
    save_path=results_dir / 'metrics_comparison.png'
)
plt.show()

print("\n📊 Métriques:")
for method, values in metrics.items():
    print(f"\n{method}:")
    for metric, value in values.items():
        print(f"  {metric}: {value:.3f}")

## 🎯 Conclusions et Recommandations

### 🔍 Comparaison des Méthodes

| Méthode | Avantages | Inconvénients | Usage recommandé |
|---------|-----------|---------------|------------------|
| **Grad-CAM** | • Rapide<br>• Précis pour CNN<br>• Facile à interpréter | • Spécifique aux CNN<br>• Résolution limitée | Analyse rapide, démo en temps réel |
| **LIME** | • Model-agnostic<br>• Explications locales<br>• Super-pixels interprétables | • Lent<br>• Dépend de la segmentation | Expliquer des cas spécifiques |
| **SHAP** | • Théoriquement fondé<br>• Cohérent<br>• Niveau pixel | • Très lent<br>• Nécessite background data | Analyse approfondie, recherche |

### 💡 Recommandations

1. **Pour la production**: Utiliser **Grad-CAM** (rapide, efficace)
2. **Pour l'analyse exploratoire**: Combiner **Grad-CAM + LIME**
3. **Pour la recherche**: Utiliser **SHAP** pour des analyses rigoureuses
4. **Pour expliquer aux médecins**: **Grad-CAM** (plus visuel et intuitif)

### 🔬 Insights médicaux

- Les modèles se concentrent principalement sur les **poumons**
- Les zones d'opacité sont fortement pondérées pour COVID
- Les patterns de réseau réticulaire sont importants
- Attention à l'**overfitting** sur des artefacts (annotations, marques)

### 📚 Références

- **Grad-CAM**: Selvaraju et al. (2017) - "Grad-CAM: Visual Explanations from Deep Networks"
- **LIME**: Ribeiro et al. (2016) - "Why Should I Trust You?"
- **SHAP**: Lundberg & Lee (2017) - "A Unified Approach to Interpreting Model Predictions"

## 📦 Export des Résultats

Tous les résultats ont été sauvegardés dans:
```
results/interpretability/
├── gradcam_covid_example.png
├── gradcam_layer_comparison.png
├── gradcam_grid.png
├── lime_example.png
├── lime_boundaries.png
├── lime_features.png
├── lime_segmentation_comparison.png
├── shap_example.png
├── shap_heatmap.png
├── shap_classes_comparison.png
├── comparison_all_methods.png
├── metrics_comparison.png
└── reports/
    ├── image_000_comparison.png
    ├── image_000_gradcam.npz
    └── image_000_report.json
```